In [26]:
# %% make repo root importable (so `from src...` works)
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root on path:", ROOT)

root on path: /Users/admin/Desktop/carbon-portfolio-project-v2


In [27]:
# %% auto-reload edited src modules (so src/*.py edits take effect without kernel restart)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
import pandas as pd
from src.db import connect
from src.data_download import batch_download_strict_min_points
from src.price_pull import prep_symbols, load_prices, build_symbol_coverage
from src.fx_pull import currencies_in_universe, fetch_fx, load_fx
from src.actions_pull import fetch_actions, load_actions

In [29]:
con = connect(str(ROOT / 'data' / 'carbon.db'))

In [30]:

# %% one-time: apply new prices/fx_rates schema (prices is empty -> safe to drop)
con.executescript((ROOT / 'sql' / 'schema.sql').read_text())


### Stock price pulls

In [55]:

# %% Section 1 — symbol prep (inspect, confirm before pulling)
ok, unmapped = prep_symbols(con)

tickers total     : 8,179
mapped to yahoo   : 6,406
unmapped exchange : 1,773
currency mix      :
currency
EUR    2647
GBp    1268
PLN     695
SEK     660
RON     294
NOK     243
CHF     204
USD     146
DKK     134
HUF      62
CZK      21
AUD      13
CAD       7
HKD       5
ZAc       3
ISK       2
ILA       2
Unmapped Exchange:
 exchange
Delisted                        724
Bulgarian Stock Exchange        242
Spotlight Stock Market          131
Nordic Growth Market (NGM)       93
Mercado Alternativo Bursatil     91
Cyprus Stock Exchange            85
Zagreb Stock Exchange            65
Malta Stock Exchange             53
OTC Bulletin Board               37
Norvegian OTC                    36


In [39]:
# %% SMOKE TEST — 50 symbols only, confirm the path before the full backfill
symbol_to_id  = dict(zip(ok['yahoo_symbol'], ok['company_id']))
symbol_to_ccy = dict(zip(ok['yahoo_symbol'], ok['currency']))

smoke = ok['yahoo_symbol'].head(50).tolist()

smoke_df, smoke_short, smoke_never = batch_download_strict_min_points(
    tickers    = smoke,
    start      = "2013-01-01", end = "2025-06-30",
    auto_adjust= False,
    batch_size = 50, pause = 20, min_count = 6,
    output_csv = None, no_data_csv = None,
)
load_prices(con, smoke_df, symbol_to_id, symbol_to_ccy)

Total tickers to process: 50

Tickers with < 6 non-NaN Close values: []

Final: 50 tickers with >= 6 data, 0 with too little data, 0 never returned any data.
prices upserted: 162,000 rows, 50 companies


In [60]:
# %% Section 2 — price backfill (auto_adjust=False -> raw prices)
filtered_df, too_short, never_seen = batch_download_strict_min_points(
    tickers    = ok['yahoo_symbol'].tolist(),
    start      = "2013-01-01", end = "2026-06-30",
    auto_adjust= False,
    batch_size = 50, pause = 20, min_count = 6,
    output_csv = "prices_backfill.csv",
    no_data_csv= "prices_missing.csv",
)



Total tickers to process: 6406


$094124453.BR: possibly delisted; no timezone found

1 Failed download:
['094124453.BR']: possibly delisted; no timezone found


$ELN.ST: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "No data found, symbol may be delisted")
$TALD.RO: possibly delisted; no timezone found
$CDSYS.BD: possibly delisted; no timezone found
$MLAST.PA: possibly delisted; no timezone found
$RY8.DU: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")

5 Failed downloads:
['ELN.ST']: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "No data found, symbol may be delisted")
['TALD.RO', 'CDSYS.BD', 'MLAST.PA']: possibly delisted; no timezone found
['RY8.DU']: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")


$K2LT.VS: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356991200, endDate = 1782766800")
$VSC.F: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "No data found, symbol may be delisted")
$4SCI.VI: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")
$MLJDL.PA: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30)
$AUXI.RO: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356991200, endDate = 1782766800")
$GREEN.RO: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356991200, endDate = 1782766800")
$PAC.WA: possibly delisted; no timezone found
$PHLOG-B.ST: possibly delisted; no timezone found
$HELIO.ST: possibly deli

$BTF.WA: possibly delisted; no timezone found
$MLIPO.PA: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30)
$TNT.WA: possibly delisted; no timezone found
$VVPR: possibly delisted; no timezone found
$FMAR.RO: possibly delisted; no timezone found
$VALBERG.BD: possibly delisted; no timezone found
$DUR.AT: possibly delisted; no timezone found
$SCIBH.MC: possibly delisted; no timezone found
$EGN.MI: possibly delisted; no timezone found

9 Failed downloads:
['BTF.WA', 'TNT.WA', 'VVPR', 'FMAR.RO', 'VALBERG.BD', 'DUR.AT', 'SCIBH.MC', 'EGN.MI']: possibly delisted; no timezone found
['MLIPO.PA']: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30)


...

$WSO.PR: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")

1 Failed download:
['WSO.PR']: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")


$EEG1T.TL: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356991200, endDate = 1782766800")
$MLHMC.PA: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")
$RLT.MI: possibly delisted; no timezone found
$EDI.PA: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "No data found, symbol may be delisted")

4 Failed downloads:
['EEG1T.TL']: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356991200, endDate = 1782766800")
['MLHMC.PA']: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")
['RLT.MI']: possibly delisted; no timezone found
['EDI.PA']: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-

$UBK.MU: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")
$ACC.L: possibly delisted; no timezone found
$ROBA.AS: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")

3 Failed downloads:
['UBK.MU', 'ROBA.AS']: possibly delisted; no price data found  (1d 2013-01-01 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1356994800, endDate = 1782770400")
['ACC.L']: possibly delisted; no timezone found


$GOODWILLPHRM.BD: possibly delisted; no timezone found

1 Failed download:
['GOODWILLPHRM.BD']: possibly delisted; no timezone found



Tickers with < 6 non-NaN Close values: ['01156.HK', '01910.HK', '01913.HK', '02233.HK', '06160.HK', '094124453.BR', '0K7K.L', '0M0.SG', '0VR2.L', '27N.DU', '3D6.BE', '4SCI.VI', '5OZ.DU', '5PG.CO', '67R.DU', '7J5.DU', '8TRA.BE', '93M.F', '9RP.SG', 'A1M.ST', 'A283WQ.DU', 'A41BVS.DU', 'A8N.MU', 'AA4.L', 'AAAP.L', 'AAQ.F', 'AAS.WA', 'ABFAST.ST', 'ABL.OL', 'ACC.L', 'ADIG.L', 'AEGA.OL', 'AEIN.SG', 'AFC.L', 'AFH.WA', 'AGCM.RO', 'AGU.MI', 'AGV.RO', 'AGXTF.L', 'ALCAB.PA', 'ALCQ.RO', 'ALCYB.PA', 'ALEVA.BR', 'ALGW.L', 'ALIMG.BR', 'ALKLN.MI', 'ALPOU.PA', 'ALREW.PA', 'ALTTU.PA', 'ALVER.PA', 'ALVG.PA', 'AME.WA', 'ANA.L', 'ANCR.L', 'ANIM.RO', 'AO9.MU', 'AQT.WA', 'ARB.DU', 'ASTG-MTF-B.ST', 'ATO.WA', 'ATOMT.PR', 'AUXI.RO', 'BADE.RO', 'BBH.L', 'BEG.L', 'BER.WA', 'BEZVA-PR.PR', 'BFV.BE', 'BGB.MU', 'BIDS.L', 'BIG.WA', 'BIG1.F', 'BIGGEORGE.BD', 'BIKE.BE', 'BKHT.BE', 'BLF.WA', 'BLV.PA', 'BOD.L', 'BONEH.HE', 'BPU.SG', 'BRH.WA', 'BSA.WA', 'BTF.RO', 'BTF.WA', 'BUF.F', 'BVX.L', 'BWIDL.OL', 'C21.L', 'C3RY.F', '

In [61]:
# %% Section 2b — load prices
symbol_to_id  = dict(zip(ok['yahoo_symbol'], ok['company_id']))
symbol_to_ccy = dict(zip(ok['yahoo_symbol'], ok['currency']))
load_prices(con, filtered_df, symbol_to_id, symbol_to_ccy)


prices upserted: 20,599,184 rows, 5,883 companies


In [75]:
filtered_df.head()

Price,Date,Ticker,Adj Close,Close,High,Low,Open,Volume
0,2013-01-01,3IN.L,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-01,A2A.MI,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-01-01,AC5.MI,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-01-01,ACE.MI,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-01-01,AMCR,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
print(pd.read_sql("""
    SELECT COUNT(*) rows,
           COUNT(DISTINCT company_id) firms,
           MIN(date) first_date, MAX(date) last_date
    FROM prices
""", con).to_string(index=False))

    rows  firms first_date  last_date
20599184   5883 2013-01-01 2026-06-29


In [64]:
# Create a new table to indicate which companies are mapped
# %% create symbol_coverage (standalone — does NOT touch prices)
con.executescript("""
CREATE TABLE IF NOT EXISTS symbol_coverage (
    company_id   TEXT PRIMARY KEY,
    ticker       TEXT, exchange TEXT, yahoo_symbol TEXT, currency TEXT,
    status       TEXT, universe TEXT,
    FOREIGN KEY (company_id) REFERENCES master_company_list(company_id)
);
""")

In [69]:
cov = build_symbol_coverage(con, ok, unmapped)

symbol_coverage: 8,288 rows
status
mapped_loaded        5883
unmapped_exchange    1773
mapped_no_data        523
no_ticker             109


In [76]:
cov_counts = pd.read_sql("""
    SELECT company_id,
           COUNT(*)                                    AS n_rows,
           SUM(CASE WHEN close IS NOT NULL THEN 1 END) AS n_real
    FROM prices
    WHERE date >= '2019-01-01'
    GROUP BY company_id
""", con)

print(f"firms: {len(cov_counts):,}")
print("\nn_real (non-null closes) distribution:")
print(cov_counts['n_real'].describe(percentiles=[.01,.05,.10,.25,.50]).to_string())
print(f"\nfirms with < 250 real: {(cov_counts['n_real'] < 250).sum():,}")
print(f"firms with < 500 real: {(cov_counts['n_real'] < 500).sum():,}")
print(f"\nrow-count vs real-count gap (padding):")
print((cov_counts['n_rows'] - cov_counts['n_real']).describe().to_string())

firms: 5,883

n_real (non-null closes) distribution:
count    5862.000000
mean     1679.153702
std       435.856958
min         6.000000
1%        176.610000
5%        546.050000
10%      1030.100000
25%      1848.250000
50%      1891.000000
max      1918.000000

firms with < 250 real: 100
firms with < 500 real: 239

row-count vs real-count gap (padding):
count    5862.000000
mean      261.273456
std       435.812186
min         5.000000
25%        42.000000
50%        54.000000
75%        91.750000
max      1935.000000


What padding is:

- when yf.download pulls a batch, it builds one shared date index = the union of every trading day any ticker in that batch traded. Any ticker that didn't trade on a given day gets a NaN row on that day. So a firm that IPO'd in 2021, or barely trades, or whose exchange had a holiday, gets NaN-filled rows. Your row-count query counted those NaN rows as if they were real data. The row-vs-real gap (median 54, max 1,935) is the padding, measured directly.

Some observations for n_real (non-null rows):
- A firm that IPO'd in 2022 has ~900 real days, all contiguous --> need to keep
- A firm that's illiquid/suspended has ~900 real days scattered with holes throughout (unreliable or highly illiquid firms)

Thus, the metric that is more informative is density = real days ÷ days it could have traded while listed.

**Late-IPO firm ≈ 1.0 (keep); gappy firm ≈ 0.5 (drop); filter on density not raw count**


In [5]:
# %% remove NaN-close padding rows from prices (reversible from backfill CSV)
before = pd.read_sql("SELECT COUNT(*) n FROM prices", con)['n'][0]
con.execute("DELETE FROM prices WHERE close IS NULL")
con.commit()
after = pd.read_sql("SELECT COUNT(*) n FROM prices", con)['n'][0]
print(f"deleted {before-after:,} NaN rows;  {after:,} real rows remain")

deleted 0 NaN rows;  15,654,648 real rows remain


In [78]:
# %% per-firm coverage + density in the 2019+ window
diag = pd.read_sql("""
    SELECT company_id, COUNT(*) AS n_real,
           MIN(date) AS first_real, MAX(date) AS last_real
    FROM prices WHERE date >= '2019-01-01'
    GROUP BY company_id
""", con)

In [79]:
# market calendar = every date any firm really traded, in window
mkt = pd.read_sql("SELECT DISTINCT date FROM prices WHERE date >= '2019-01-01'",
                  con)['date'].sort_values().reset_index(drop=True)

mkt_idx = {d: i for i, d in enumerate(mkt)}

# expected days = market-calendar days between a firm's first and last real trade
diag['expected'] = diag.apply(
    lambda r: mkt_idx[r['last_real']] - mkt_idx[r['first_real']] + 1, axis=1)
diag['density'] = diag['n_real'] / diag['expected']

print(f"firms with real data in window: {len(diag):,}")
print("\ndensity distribution:")
print(diag['density'].describe(percentiles=[.01,.05,.10,.25]).to_string())
print(f"\ndensity < 0.90: {(diag['density'] < 0.90).sum():,}")
print(f"density < 0.95: {(diag['density'] < 0.95).sum():,}")
print(f"n_real   < 500: {(diag['n_real'] < 500).sum():,}")
print("\nlate starters (first_real > 2020, but dense >0.95):")
print((( diag['first_real'] > '2020-01-01') & (diag['density'] > 0.95)).sum())

firms with real data in window: 5,862

density distribution:
count    5862.000000
mean        0.963620
std         0.057962
min         0.030769
1%          0.878555
5%          0.952539
10%         0.958585
25%         0.962936
50%         0.969247
max         1.000000

density < 0.90: 63
density < 0.95: 147
n_real   < 500: 239

late starters (first_real > 2020, but dense >0.95):
1178


In [80]:
elig = (diag['density'] >= 0.90) & (diag['n_real'] >= 250)
print(f"eligible:   {elig.sum():,}")
print(f"excluded:   {(~elig).sum():,}")
print("\nexcluded breakdown:")
print(f"  fails density (<0.90): {(diag['density'] < 0.90).sum():,}")
print(f"  fails n_real  (<250):  {(diag['n_real'] < 250).sum():,}")
print(f"  fails both:            {((diag['density']<0.90) & (diag['n_real']<250)).sum():,}")

# peek at what we'd drop — sanity check they're really junk
print("\nsample of excluded firms:")
print(diag[~elig].merge(
    pd.read_sql('SELECT company_id, company_name, exchange FROM master_company_list', con),
    on='company_id')[['company_name','exchange','n_real','density']]
    .sort_values('density').head(15).to_string(index=False))

eligible:   5,711
excluded:   151

excluded breakdown:
  fails density (<0.90): 63
  fails n_real  (<250):  100
  fails both:            12

sample of excluded firms:
                      company_name                   exchange  n_real  density
                       FIMARO S.A.   Bucharest Stock Exchange      16 0.030769
                     SC ALBAPAM SA   Bucharest Stock Exchange      30 0.057692
                        PRACTIC SA   Bucharest Stock Exchange      52 0.096296
CONSTRUCTII FEROVIARE CRAIOVA S.A.   Bucharest Stock Exchange      67 0.117958
    INVERSIONES HERRERO, SA, SICAV            Bolsa de Madrid     177 0.119595
                        COMALEX SA   Bucharest Stock Exchange      77 0.143657
     GSTAAD INVERSIONES, SICAV, SA Bolsa de Valores de Bilbao     241 0.156291
               SIDICLEAR SICAV SA.            Bolsa de Madrid     342 0.267396
   PENSIONINVEST CAPITAL SICAV SA. Bolsa de Valores de Bilbao     297 0.268293
       FONDUL OAMENILOR DE AFACERI   Buchar

In [83]:
# %% add eligibility flag to symbol_coverage (post-pull filter: density>=0.90 AND n_real>=250)
con.execute("ALTER TABLE symbol_coverage ADD COLUMN eligible INTEGER DEFAULT 0")

# create a set of company id with eligible data:
elig_ids = set(diag[(diag['density'] >= 0.90) & (diag['n_real'] >= 250)]['company_id'])

# populate rows in database (only matched eligible company ids):
con.executemany("UPDATE symbol_coverage SET eligible=? WHERE company_id=?",
                [(1 if c in elig_ids else 0, c) for c in
                 pd.read_sql("SELECT company_id FROM symbol_coverage", con)['company_id']])
con.commit()


In [84]:
# 
print(pd.read_sql("""
    SELECT status, eligible, COUNT(*) n
    FROM symbol_coverage GROUP BY status, eligible ORDER BY status, eligible
""", con).to_string(index=False))

           status  eligible    n
    mapped_loaded         0  172
    mapped_loaded         1 5711
   mapped_no_data         0  523
        no_ticker         0  109
unmapped_exchange         0 1773


In [91]:
# the 5,711-firm analysis panel, any time you need it
panel = pd.read_sql("""
    SELECT p.company_id, p.date, p.close, p.currency, p.high, p.low, p.open, p.volume
    FROM prices p
    JOIN symbol_coverage s ON s.company_id = p.company_id
    WHERE s.eligible = 1 AND p.date >= '2019-01-01'
""", con)

### Fx Pull

why do we need fx? 
Pulled prices are in local currency, and we aim to convert to common currency (EUR); we need to adopt a by-day fx rate for read-time conversion.

Goal: one EUR exchange rate per (currency, date) so read-time conversion can turn local prices into EUR. eur_price = local_price / rate_per_eur.
There is a total currency mix of ~17, but EUR itself needs no rate (it's the base), so we pull the ~13 non-EUR ones.

In [95]:
# BP plc: ~£4-5/share. If close is ~450, it's pence (GBp). If ~4.5, it's pounds.
chk = pd.read_sql("""
    SELECT s.company_id, p.date, p.close, p.currency
    FROM prices p JOIN symbol_coverage s ON s.company_id=p.company_id
    WHERE s.ticker IN ('BP.','ULVR','AZN') AND p.date >= '2026-06-01'
    ORDER BY p.date DESC LIMIT 10
""", con)

print(chk.to_string(index=False))

  company_id       date        close currency
GB00BVZK7T90 2026-06-29  4580.000000      GBp
GB0007980591 2026-06-29   472.700012      GBp
GB0009895292 2026-06-29 14298.000000      GBp
GB00BVZK7T90 2026-06-26  4600.500000      GBp
GB0007980591 2026-06-26   469.399994      GBp
GB0009895292 2026-06-26 14318.000000      GBp
GB00BVZK7T90 2026-06-25  4579.500000      GBp
GB0007980591 2026-06-25   480.850006      GBp
GB0009895292 2026-06-25 14062.000000      GBp
GB00BVZK7T90 2026-06-24  4544.000000      GBp


In [98]:
ccys = currencies_in_universe(con)
print(f"{len(ccys)} currencies:", ccys)


15 currencies: ['AUD', 'CAD', 'CHF', 'CZK', 'DKK', 'GBP', 'HUF', 'ILS', 'ISK', 'NOK', 'PLN', 'RON', 'SEK', 'USD', 'ZAR']


In [99]:
# %% Section 3b — pull FX (~13 symbols, a minute or two)
fx_df = fetch_fx(ccys)
print(f"\npulled {len(fx_df):,} rows")
print(fx_df.groupby('currency')['date'].agg(['count', 'min', 'max']).to_string())


pulled 52,689 rows
          count         min         max
currency                               
AUD        3513  2013-01-01  2026-06-29
CAD        3513  2013-01-01  2026-06-29
CHF        3513  2013-01-01  2026-06-29
CZK        3513  2013-01-01  2026-06-29
DKK        3513  2013-01-01  2026-06-29
GBP        3513  2013-01-01  2026-06-29
HUF        3513  2013-01-01  2026-06-29
ILS        3513  2013-01-01  2026-06-29
ISK        3511  2013-01-01  2026-06-29
NOK        3513  2013-01-01  2026-06-29
PLN        3513  2013-01-01  2026-06-29
RON        3511  2013-01-01  2026-06-29
SEK        3513  2013-01-01  2026-06-29
USD        3511  2013-01-01  2026-06-29
ZAR        3513  2013-01-01  2026-06-29


In [100]:
# %% Section 3c — load into fx_rates
load_fx(con, fx_df)


fx upserted: 52,689 rows, 15 currencies


In [101]:
# sanity: row counts + a spot rate per currency
print(pd.read_sql("""
    SELECT currency, COUNT(*) n, MIN(date) first, MAX(date) last
    FROM fx_rates GROUP BY currency ORDER BY currency
""", con).to_string(index=False))

currency    n      first       last
     AUD 3513 2013-01-01 2026-06-29
     CAD 3513 2013-01-01 2026-06-29
     CHF 3513 2013-01-01 2026-06-29
     CZK 3513 2013-01-01 2026-06-29
     DKK 3513 2013-01-01 2026-06-29
     GBP 3513 2013-01-01 2026-06-29
     HUF 3513 2013-01-01 2026-06-29
     ILS 3513 2013-01-01 2026-06-29
     ISK 3511 2013-01-01 2026-06-29
     NOK 3513 2013-01-01 2026-06-29
     PLN 3513 2013-01-01 2026-06-29
     RON 3511 2013-01-01 2026-06-29
     SEK 3513 2013-01-01 2026-06-29
     USD 3511 2013-01-01 2026-06-29
     ZAR 3513 2013-01-01 2026-06-29


In [102]:
print(pd.read_sql("""
    SELECT date, rate_per_eur FROM fx_rates
    WHERE currency='GBP' ORDER BY date DESC LIMIT 3
""", con).to_string(index=False))

      date  rate_per_eur
2026-06-29       0.86270
2026-06-26       0.86133
2026-06-25       0.86245


### Splits Data

note: we have indicated auto_adjust to Flase when pulling price data via yahoo finance. This is because prices oulled with auto adjusting by default would not update prior downloaded data that have been stored in our database. INSERT OR IGNORE would never rewrite the old rows → silent drift.

Raw prices never change. €50.20 on 2013-05-01 is €50.20 forever. So storing raw makes INSERT OR IGNORE genuinely idempotent, and we compute the adjustment fresh at read-time from a separate splits table


Feature engineering flow:
- store raw OHLCV (done) + store split events (this step)
- apply the adjustment when you pull the panel (feature stage)
- The splits table is small; most firms have zero or one split in the window — so this is cheap.

adjustment methods
- back-adjust the pre-split prices by the split ratio so the series is continuous
- Multiply every price before the split by the ratio's inverse:

In [31]:
# %% create corporate_actions table
con.executescript("""
CREATE TABLE IF NOT EXISTS corporate_actions (
    company_id  TEXT NOT NULL, date TEXT NOT NULL, action_type TEXT NOT NULL,
    value REAL, source TEXT DEFAULT 'yfinance',
    PRIMARY KEY (company_id, date, action_type),
    FOREIGN KEY (company_id) REFERENCES master_company_list(company_id)
);
""")
print("corporate_actions created")

corporate_actions created


In [32]:
# %% Section 4a — build the symbol->id map for ELIGIBLE firms only
from src.actions_pull import fetch_actions, load_actions

elig_syms = pd.read_sql("""
    SELECT yahoo_symbol, company_id
    FROM symbol_coverage
    WHERE eligible = 1 AND yahoo_symbol IS NOT NULL
""", con)

symbol_to_id = dict(zip(elig_syms['yahoo_symbol'], elig_syms['company_id']))

print(f"pulling actions for {len(symbol_to_id):,} eligible firms")

pulling actions for 5,711 eligible firms


In [34]:
actions_df = fetch_actions(symbol_to_id,
                           checkpoint_csv=str(ROOT / 'data' / 'raw' / 'actions_checkpoint.csv'))

to fetch: 5,711 firms

  620/5711  (10%)  actions so far: 10,163

AEDES.MI: auto_adjust failed with unsupported operand type(s) for /: 'str' and 'float'


  5711/5711  (100%)  actions so far: 79,022
failed for 4 symbols (first 10): ['YIRG.MC', 'PRE.WA', 'SCAML.MC', 'TTS.RO']

total: 79,022 actions: 2,823 splits, 76,199 dividends


In [42]:
actions_df

,company_id,date,action_type,value
0,AT000000STR1,2008-06-27,dividend,0.55
1,AT000000STR1,2009-06-26,dividend,0.55
2,AT000000STR1,2010-06-25,dividend,0.50
3,AT000000STR1,2011-06-17,dividend,0.55
4,AT000000STR1,2012-06-22,dividend,0.60
...,...,...,...,...
79017,SE0017083272,2025-08-15,dividend,0.16
79018,SE0017083272,2025-11-14,dividend,0.16
79019,SE0017083272,2026-02-13,dividend,0.16
79020,SE0017083272,2026-05-21,dividend,0.18


In [35]:
# %% Section 4c — load + sanity
load_actions(con, actions_df)

actions upserted: 79,022 rows, 4,205 companies


In [39]:
print(pd.read_sql("""
    SELECT action_type, COUNT(*) n, COUNT(DISTINCT company_id) firms
    FROM corporate_actions
    GROUP BY action_type
""", con))

  action_type      n  firms
0    dividend  76199   3769
1       split   2823   1732


In [43]:
# 1. Split ratios should be sensible (mostly 2.0, 3.0, 0.5 for reverse, etc.)
print(pd.read_sql("""
    SELECT value AS split_ratio, COUNT(*) n
    FROM corporate_actions WHERE action_type='split'
    GROUP BY value ORDER BY n DESC LIMIT 10
""", con).to_string(index=False))

 split_ratio   n
        2.00 393
        0.10 263
        5.00 210
       10.00 190
        4.00 184
        3.00 150
        0.01 140
        1.10  78
        0.20  70
        0.05  65


In [44]:
# 2. A known splitter as a spot-check — any large firm you recognize
print(pd.read_sql("""
    SELECT m.company_name, c.date, c.value
    FROM corporate_actions c
    JOIN master_company_list m ON m.company_id = c.company_id
    WHERE c.action_type='split'
    ORDER BY c.date DESC LIMIT 8
""", con).to_string(index=False))


                        company_name       date     value
                11 88 0 SOLUTIONS AG 2026-08-10  0.200000
               ACCELER8 VENTURES PLC 2026-08-07  4.041100
                 SUNDELL ESTATE NYRT 2026-08-04 20.000000
                      REDCENTRIC PLC 2026-07-29  0.050000
                   BANCA PROFILO SPA 2026-07-27  0.100000
PARTNER AEROSPACE & DEFENSE GROUP SA 2026-07-27  0.010000
        AGROLAND BUSINESS SYSTEM S.A 2026-07-27  1.031250
                 TRINITY BIOTECH PLC 2026-07-24  0.033333
